In [ ]:
!pip install git+https://github.com/google-research/perch-hoplite.git
!pip install tensorflow[and-cuda]~=2.20

  Cloning https://github.com/google-research/perch-hoplite.git to /tmp/pip-req-build-qfxyral7
  Running command git clone --filter=blob:none --quiet https://github.com/google-research/perch-hoplite.git /tmp/pip-req-build-qfxyral7
  Resolved https://github.com/google-research/perch-hoplite.git to commit 8c4d3d1d7ccb49b5d1764f5c0a2ddb9e92721739
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.9/77.9 kB 5.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.6/85.6 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.3/68.3 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 109.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 99.7 MB/s eta 0:00:00
  

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.6/572.6 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 93.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 293.6/293.6 MB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.3/324.3 kB 32.6 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.6
    Uninstalling protobuf-5.29.6:
      Successfully uninstalled protobuf-5.29.6
  Attempting uninstall: nvidia-nccl-cu12
    Found existing installation: nvidia-nccl-cu12 2.27.5
    Uninstalling nvidia-nccl-cu12-2.27.5:
      Successfully uninstalled nvidia-nccl-cu12-2.27.5
  Attempting uninstall: h5py
    Found existing installation: h5py 3.16.0
    Uninstalling h5py-3.16.0:
      Successfully uninstalled h5py-3.16.0
  Attempting uninstall: tensorflow
    Found existing installation: tensorflow 2.19.0
    Uninstalling tensorflow-2.19.0:
ERROR: Operation cancelled by user
^C


## Config

In [ ]:
from etils import epath
import pandas as pd
from ml_collections import config_dict
import numpy as np
import os

from perch_hoplite.agile import colab_utils, embed, source_info
from perch_hoplite.db import brutalism, interface
from perch_hoplite.agile import audio_loader, classifier, classifier_data
from perch_hoplite.agile import embedding_display, source_info
from perch_hoplite.db import brutalism, score_functions, search_results
from perch_hoplite.db import sqlite_usearch_impl
from perch_hoplite.zoo import model_configs, taxonomy_model_tf

# Your config
dataset_name = 'puma_recordings'
dataset_base_path = 'H:/Perch data/output/'
dataset_fileglob = '*.wav'
db_path = 'H/Perch data/puma_hoplite_db'
model_choice = 'perch_v2'  # good multi-taxa model, works for mammals
annotator_id = 'puma_annotator'

## Embedding

In [ ]:
audio_glob = source_info.AudioSourceConfig(
    dataset_name=dataset_name,
    base_path=dataset_base_path,
    file_glob=dataset_fileglob,
    min_audio_len_s=1.0,
    target_sample_rate_hz=-2,  # uses model's native rate
    shard_len_s=None,  # no sharding for short files
)

configs = colab_utils.load_configs(
    source_info.AudioSources((audio_glob,)),
    db_path,
    model_config_key=model_choice,
    db_key='sqlite_usearch',
)

db = configs.db_config.load_db()

worker = embed.EmbedWorker(
    audio_sources=configs.audio_sources_config,
    db=db,
    model_config=configs.model_config,
)

worker.process_all(target_dataset_name=audio_glob.dataset_name)
print('Total embeddings:', db.count_embeddings())


Adding deployments...


100%|██████████| 191/191 [00:00<00:00, 3327.43it/s]



Adding recordings...


100%|██████████| 191/191 [00:00<00:00, 8985.11it/s]



Adding annotations...

Embedding audio...


100%|██████████| 191/191 [00:45<00:00,  4.19it/s]


Total embeddings: 191


## Training

In [ ]:
db = sqlite_usearch_impl.SQLiteUSearchDB.create(db_path)
db_model_config = db.get_metadata('model_config')
embed_config = db.get_metadata('audio_sources')
model_class = model_configs.get_model_class(db_model_config.model_key)
embedding_model = model_class.from_config(db_model_config.model_config)
audio_sources = source_info.AudioSources.from_config_dict(embed_config)

window_size_s = getattr(embedding_model, 'window_size_s', 5.0)
audio_filepath_loader = audio_loader.make_filepath_loader(
    audio_sources=audio_sources,
    window_size_s=window_size_s,
    sample_rate_hz=embedding_model.sample_rate,
)

In [ ]:
# Load your master CSV
labels_df = pd.read_csv('C:/Users/Luis Rouzaud/Documents/Perch detection/output/master_metadata.csv')

# Get all window IDs and their source info from the DB
all_window_ids = db.match_window_ids()
annotations = []

print(vars(db.get_window(all_window_ids[0])))
recording = db.get_recording(1)
print(vars(recording))
print([x for x in dir(interface.datatypes) if not x.startswith('_')])

for window_id in all_window_ids:
    window = db.get_window(window_id)
    recording = db.get_recording(window.recording_id)
    filename = recording.filename

    match = labels_df[labels_df['window_filename'] == filename]
    if not match.empty:
        label = match.iloc[0]['label']
        annotations.append(
            interface.datatypes.Annotation(
                id=window_id,
                window_id=window_id,
                recording_id=window.recording_id,
                offsets=window.offsets,
                label=label,
                label_type=interface.datatypes.LabelType.POSITIVE,
                annotator_id=annotator_id,
                is_positive=True,
                provenance='csv_import',
            )
        )

db.insert_annotations(annotations, handle_duplicates='skip')
db.commit()
print(f'Ingested {len(annotations)} annotations')
print(f'Total annotations in DB: {len(db.get_all_annotations())}')

In [ ]:
target_labels = None  # auto-populated from DB annotations
learning_rate = 1e-3
weak_neg_weight = 0.05
l2_mu = 0.0
num_steps = 128
train_ratio = 0.9
batch_size = 128
weak_negatives_batch_size = 128

data_manager = classifier_data.AgileDataManager(
    target_labels=target_labels,
    db=db,
    train_ratio=train_ratio,
    min_eval_examples=1,
    batch_size=batch_size,
    weak_negatives_batch_size=weak_negatives_batch_size,
    rng=np.random.default_rng(seed=5),
)

print('Training for target labels:')
print(data_manager.get_target_labels())

linear_classifier, eval_scores = classifier.train_linear_classifier(
    data_manager=data_manager,
    learning_rate=learning_rate,
    weak_neg_weight=weak_neg_weight,
    num_train_steps=num_steps,
)

print(f"\ntop-1:   {eval_scores['top1_acc']:.3f}")
print(f"roc_auc: {eval_scores['roc_auc']:.3f}")
print(f"cmap:    {eval_scores['cmap']:.3f}")

linear_classifier.save(os.path.join(db_path, 'puma_classifier.pt'))

Training for target labels:
('moving', 'resting')


Loss 0.05098850: 100%|██████████| 128/128 [00:07<00:00, 18.23it/s]


top-1:   0.944
roc_auc: 1.000
cmap:    1.000


## Inference

In [ ]:
output_csv = 'C:/Users/Luis Rouzaud/Documents/Perch detection/puma_hoplite_db/inference.csv'
logit_threshold = 1.0  # adjust based on your precision/recall needs

classifier.write_inference_csv(
    linear_classifier,
    db,
    output_csv,
    logit_threshold,
    labels=None,
    window_ids=db.match_window_ids(),
)